In [ ]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [ ]:
import pandas as pd

In [ ]:
from notebooks.radp_library import preprocess_ue_data, mro_plot_scatter, plot_sinr_db_by_ue
from radp.digital_twin.utils.cell_selection import perform_attachment, perform_attachment_hyst_ttt
from notebooks.plots import *

In [ ]:
ue_data = pd.read_csv("data/mro_data/UE_data_20UE_100ticks.csv")
ue_data

In [ ]:
topology = pd.read_csv("data/mro_data/mro_topology.csv")
topology

In [ ]:
preprocessed_data = preprocess_ue_data(ue_data, topology)
preprocessed_data = preprocessed_data.rename(columns={'latitude': 'lat', 'longitude': 'lon', 'cell_rxpwr_dbm': 'rxpower_dbm', 'mock_ue_id': 'ue_id'})
preprocessed_data

In [ ]:
attached_df = perform_attachment(preprocessed_data, topology)
attached_df

In [ ]:
topology['cell_id'] = topology['cell_id'].astype(str).str.extract(r'(\d+)').astype(float)

In [ ]:
plot_naive_attached_df(attached_df, topology)

### OPTIMIZED ATTACHMENT

In [ ]:
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO
from apps.mobility_robustness_optimization.mobility_robustness_optimization import calculate_mro_metric
from plots import add_sinr_db
mro = SimpleMRO({}, topology)
from radp.digital_twin.utils.constants import RLF_THRESHOLD
from radp.digital_twin.utils.cell_selection import find_hyst_diff

In [ ]:
ue_data = pd.read_csv("data/mro_data/UE_data_20UE_100ticks.csv")
topology = pd.read_csv("data/mro_data/mro_topology.csv")
preprocessed_data = preprocess_ue_data(ue_data, topology)
preprocessed_data = preprocessed_data.rename(columns={"mock_ue_id": "ue_id", "lat": "latitude", "lon": "longitude", "cell_rxpwr_dbm": "cell_rxpower_dbm"})
preprocessed_data

In [ ]:
preprocessed_data = add_sinr_db(preprocessed_data)
preprocessed_data

In [ ]:
# epochs = 100
epochs = 50
hyst = 0.01
ttt = 5
rlf_threshold = RLF_THRESHOLD

attached_df = perform_attachment_hyst_ttt(preprocessed_data, hyst, ttt, rlf_threshold)
max_diff = find_hyst_diff(preprocessed_data)
num_ticks = preprocessed_data["tick"].nunique()
hyst_range = [0, max_diff]
ttt_range = [2, num_ticks + 1]

score = pd.DataFrame(columns=["hyst", "ttt", "score"])

header = f"{'Epoch':<6} {'Hyst':<14} {'TTT':<6} {'MRO Metric':<12}"
print(header)
print("-" * len(header))
score.loc[len(score)] = [hyst, ttt, calculate_mro_metric(attached_df)]
for i in range(epochs):
    while True:
        hyst = np.random.uniform(hyst_range[0], hyst_range[1])
        ttt = np.random.randint(ttt_range[0], ttt_range[1])
        if ttt not in score["ttt"].values or hyst not in score["hyst"].values:
            break
    # Perform attachment and calculate MRO Metric
    attached_df = perform_attachment_hyst_ttt(preprocessed_data, hyst, ttt, rlf_threshold)
    mro_metric = calculate_mro_metric(attached_df)

    # Store the data in the score DataFrame
    score.loc[len(score)] = [hyst, ttt, mro_metric]
    print(f"{i:<6} {hyst:<14.10f} {ttt:<6} {mro_metric:<12.6f}")

hyst = score.loc[score['score'].idxmax(), 'hyst']
ttt = int(score.loc[score['score'].idxmax(), 'ttt'])
print(
    f"""\nOptimized Hyst: {hyst},
    Optimized TTT: {ttt}"""
)

In [ ]:
attached_df = perform_attachment_hyst_ttt(preprocessed_data, hyst, ttt, RLF_THRESHOLD)
attached_df = attached_df.rename(columns={"latitude": "loc_y", "longitude": "loc_x",})

In [ ]:
attached_df

In [ ]:
mro_plot_scatter(attached_df, topology)

In [ ]:
for i in range(20):
    plot_sinr_db_by_ue(attached_df, preprocessed_data, i)